# N101 · 0/1背包与逆序容量

**目标：** 证明滚动数组逆序更新避免重复用物品。

**先修：** N098, N064。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 物品维度；容量；最大值/可达；恰好/至多；负无穷初始化。

**配套讲解来源：** [同名逐章意见](../../comment/101_zero_one_knapsack.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章围绕"每件东西只能拿一次"的取舍问题展开，有三个代表问题：

1. **0/1 背包**：给你若干件物品，第 i 件重 `weights[i]`、值 `values[i]`，还有一个容量为 C 的背包，每件物品要么整个装进去、要么不装，问装进来的物品总价值最大是多少。比如物品重量 `[2,3,4]`、价值 `[4,5,7]`、容量 6，答案是 `11`：装第一件和第三件（重 2+4=6，恰好塞满，价值 4+7=11），任何别的组合都不如它。
2. **分割等和子集**：问能不能把一组正整数分成两堆，使两堆的和相等。比如 `[1,5,11,5]` 返回 `True`，因为 `1+5+5=11`，正好和剩下的 11 相等。
3. **目标和**：给每个数前面添 `+` 或 `−`，问有多少种添法使整个表达式的结果等于 target。比如 `[0,0]`、target=0 返回 `4`：两个 0 各自添正号或负号都不影响结果，2×2=4 种，注意数值为 0 的元素依然有正、负两种选择，所以不能去掉重复。

这三个问题的共同骨架是：每件物品有"选"与"不选"两条岔路，我们要在容量（或和）这个维度上记录结果。

## 2. 基础知识：先读懂这些词

- **0/1 背包**：每件物品最多选一次（选 0 次或 1 次，所以叫 0/1），区别于后面章节"每件可以拿任意多件"的完全背包。
- **物品维度与容量维度**：标准二维 DP 是 `dp[i][c]` = "只允许用前 i 件物品、容量不超过 c 时的最优值"。物品维度 i 一层层推进，容量维度 c 是同一层里的各个格子。
- **滚动数组与逆序更新**：算第 i 层只需要第 i−1 层，所以能把二维压成一维 `dp[c]`。但压缩后有个坑：转移要读 `dp[c-w]`，它是"上一层"的旧值还是"本层刚更新过"的新值，取决于遍历方向。**容量必须从大到小（逆序）遍历**，这样读 `dp[c-w]` 时它还没被本层更新，仍是上一层的值，同一件物品就不会被拿两次。这是本章标题级别的重点。
- **可达性（布尔 DP / 位集）**：有的问题不关心最优值，只关心"某个和能不能凑出来"。可以开一个布尔数组标记每个和是否可达；更进一步，用一个整数的第 s 个二进制位表示"和 s 是否可达"，整数的移位和按位或就批量完成了所有格子的转移。
- **左移 `<<` 与按位或 `|`**：`reachable<<x` 把所有可达位整体左移 x 位，含义是"每个已达成的和 s 变成 s+x"，也就是把物品 x 塞进每个已有方案；`|=` 把"不拿 x（原位）"与"拿 x（移位后）"两批方案合并。
- **方案数 DP 与 Counter**：`find_target_sum_ways` 需要的不是最优值而是"达到每个中间值有多少种方式"。用字典 `Counter` 把"表达式的当前值 → 方案数"整层保存，每处理一个数就把每条状态的方案数向 `值+x` 和 `值−x` 两个方向各搬运一次，最后查目标值的方案数。
- **负无穷初始化（恰好型背包的技巧）**：如果想要求"恰好装满容量"的最优值，要把除了 0 容量外的格子初始化成负无穷，表示"这些和还凑不出来"。本章接口用的是"至多装满"语义所以初始化为 0，但知识点里提到这个区分，你要知道两者差别。
- **伪多项式复杂度**：背包的运行时间与容量 C（一个数值大小，而不是输入文本的长度）成正比。数值 C 用二进制写只需要 log C 位，所以严格说这不算对输入长度的多项式算法，叫"伪多项式"。实际使用中 C 是多少就跑多少步，仍然很快。

## 3. 思路推导：从小例子开始

**问题 1：0/1 背包（`knapsack_01`）**

- Step 1：按二维定义 `dp[i][c]`：只考虑前 i 件、容量 c 时的最大价值。转移是二选一：不拿第 i 件，答案沿用 `dp[i-1][c]`；拿第 i 件（前提 c 装得下，即 c≥w），答案是 `dp[i-1][c-w]+v`。取较大者。
- Step 2：把二维压成一维。`dp[c]` 在本轮更新前存的就是上一层的值，更新后就变本层的值。
- Step 3：关键一步——**容量从 C 往 w 倒着扫**。扫到 c 时，`dp[c-w]` 比 c 小，还没被本轮碰过，读到的必然是上一层（不含当前物品）的值，所以当前物品最多被拿一次。如果正序扫，`dp[c-w]` 已经掺入了当前物品的贡献，等于同一件物品被拿了两次，就变成完全背包了。
- Step 4：所有物品处理完，`dp[capacity]` 就是答案。

“运行示例”部分 演示了物品 `[2,3,4]`、价值 `[4,5,7]`、容量 6 的逐层表格（"各容量最优值"是容量 0..6 七个格子）：

| 已处理物品 | 各容量最优值（c=0..6） |
|---|---|
| 无物品 | 0, 0, 0, 0, 0, 0, 0 |
| w=2,v=4 | 0, 0, 4, 4, 4, 4, 4 |
| w=3,v=5 | 0, 0, 4, 5, 5, 9, 9 |
| w=4,v=7 | 0, 0, 4, 5, 7, 9, 11 |

手算第二轮（处理 w=3,v=5，c 从 6 倒着扫）验证逆序的作用：c=6 时 `dp[3]` 还是上一层的 4，得 `max(4, 4+5)=9`；c=5 时 `dp[2]` 还是 4，得 9；c=4 时 `dp[1]` 是 0，得 5；c=3 时 `dp[0]` 是 0，得 5。读到的全是旧层值。最终 `dp[6]=11`，对应"第一件 + 第三件"。

**问题 2：分割等和子集（`can_partition`）**

- Step 1：两堆相等 ⟺ 存在一个子集的和恰好等于总和的一半。所以先算总和，奇数直接返回 False。
- Step 2：问题变成"能不能从数里凑出 total//2"，这是可达性问题，用位集：整数 `reachable` 的第 s 位为 1 表示"和 s 可以凑出"。
- Step 3：初始只有第 0 位是 1（什么都不选，和为 0）。每来一个数 x，`reachable |= reachable<<x` 一句完成"不选 x"与"选 x"两种可能的合并。
- Step 4：最后看第 total//2 位是不是 1。

手算 `[1,5,11,5]`：total=22，目标 11。初始 `{0}`；加 1 得 `{0,1}`；加 5 得 `{0,1,5,6}`；加 11 得 `{0,1,5,6,11,12,16,17}`——11 已经可达，加最后一个 5 不影响结论，返回 True。

**问题 3：目标和（`find_target_sum_ways`）**

- Step 1：这是计数问题，状态是"处理完前 k 个数后，表达式能取到的每个值，各有多少种符号搭配"。
- Step 2：初始状态是"0 个数、值为 0、恰好 1 种方式"。每处理一个数 x，每个旧状态 `(值, 数量)` 分裂成 `(值+x, 数量)` 和 `(值−x, 数量)` 两条，数量各搬运一份。
- Step 3：所有数处理完，直接查 target 那一项的方案数。

手算 `[0,0]`、target=0：初始 `{0:1}`；处理第一个 0 得 `{0:2}`（+0 和 −0 是两条不同的路）；处理第二个 0 得 `{0:4}`。答案是 4——这正是 “自动测试”部分 里那条断言考察的点：值为 0 的元素也有正负两种选择。

## 4. 核心代码：knapsack_01

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def knapsack_01(weights, values, capacity):
    if len(weights) != len(values) or capacity < 0 or any((w < 0 for w in weights)):
        raise ValueError('invalid knapsack')
    dp = [0] * (capacity + 1)
    for w, v in zip(weights, values):
        for c in range(capacity, w - 1, -1):
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[-1]
```

### 逐段读懂代码

**函数一：`knapsack_01`**

```python
def knapsack_01(weights, values, capacity):
    if len(weights) != len(values) or capacity < 0 or any((w < 0 for w in weights)):
        raise ValueError('invalid knapsack')
    dp = [0] * (capacity + 1)
    for w, v in zip(weights, values):
        for c in range(capacity, w - 1, -1):
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[-1]
```

- 第一行是输入检查：重量和价值数量必须一致、容量不能为负、不能有负重量的物品，任何一条不满足就报错。这是在挡非法输入，防止转移逻辑在奇怪数据上悄悄产出错误答案。
- `dp=[0]*(capacity+1)`：下标 0..capacity 共 C+1 个格子，初始化为 0 表示"至多装满"语义下，什么都装不了价值为 0。
- `for w,v in zip(weights,values)`：一件一件物品推进"物品维度"。
- `for c in range(capacity,w-1,-1)`：容量从大到小扫到 w 为止。`w-1` 作为终点配合步长 −1，意思是最后一个处理的容量是 w（再小就装不下这件物品了）。逆序是本函数的灵魂，理由见上一节 Step 3。
- `dp[c]=max(dp[c],dp[c-w]+v)`：不拿这件（旧 dp[c]）与拿这件（上一层 dp[c-w] 加价值）取大。
- `return dp[-1]`：容量给满时的最优值。

**函数二：`can_partition`**

```python
def can_partition(nums):
    total = sum(nums)
    if any((x < 0 for x in nums)):
        raise ValueError('nonnegative values required')
    if total % 2:
        return False
    reachable = 1
    for x in nums:
        reachable |= reachable << x
    return bool(reachable >> total // 2 & 1)
```

- `if any(x<0 ...)`：负数会破坏"子集和"的位集语义（左移负数位没有意义），直接挡掉。
- `if total%2: return False`：总和是奇数就不可能平分成两个整数和，这是最快的剪枝。
- `reachable=1`：整数为 1，即只有第 0 位是 1，表示"和 0 可达（什么都不选）"。
- `reachable|=reachable<<x`：先左移得到"每个已有子集都加上 x"的新可达集，再与原来的按位或，得到"不选 x ∪ 选 x"的完整可达集。
- `bool(reachable>>(total//2)&1)`：把目标位右移到最低位，与 1 取按位与再转成布尔，就是"和为 total//2 是否可达"。

**函数三：`find_target_sum_ways`**

```python
def find_target_sum_ways(nums, target):
    states = Counter({0: 1})
    for x in nums:
        nxt = Counter()
        for value, count in states.items():
            nxt[value + x] += count
            nxt[value - x] += count
        states = nxt
    return states[target]
```

- `states=Counter({0:1})`：初始只有一个状态"值为 0，1 种方式"。
- `nxt=Counter()`：每层新建一个空计数器，旧层只读、新层只写，两层互不干扰，天然避免了原地更新可能串层的坑。
- `nxt[value+x]+=count; nxt[value-x]+=count`：每个旧状态沿"+"边和"−"边各分裂一次，方案数各加一份。注意即使 x 是 0，`value+x` 和 `value−x` 是同一个键，会加两次，这正确反映了"+0"和"−0"是两种不同的符号选择。
- `return states[target]`：目标值不在字典里时 Counter 返回 0，没有方案。

## 5. 分段实现：按顺序运行

**本章接口：** `knapsack_01(weights, values, capacity)`、`can_partition(nums)`、`find_target_sum_ways(nums, target)`

### 导入本章使用的库

In [1]:
from collections import Counter

### knapsack_01

In [2]:
def knapsack_01(weights, values, capacity):
    if len(weights) != len(values) or capacity < 0 or any((w < 0 for w in weights)):
        raise ValueError('invalid knapsack')
    dp = [0] * (capacity + 1)
    for w, v in zip(weights, values):
        for c in range(capacity, w - 1, -1):
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[-1]

### can_partition

In [3]:
def can_partition(nums):
    total = sum(nums)
    if any((x < 0 for x in nums)):
        raise ValueError('nonnegative values required')
    if total % 2:
        return False
    reachable = 1
    for x in nums:
        reachable |= reachable << x
    return bool(reachable >> total // 2 & 1)

### find_target_sum_ways

In [4]:
def find_target_sum_ways(nums, target):
    states = Counter({0: 1})
    for x in nums:
        nxt = Counter()
        for value, count in states.items():
            nxt[value + x] += count
            nxt[value - x] += count
        states = nxt
    return states[target]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

weights=[2,3,4]; values=[4,5,7]; cap=6; dp=[0]*(cap+1); rows=[('无物品',dp[:])]
for w,v in zip(weights,values):
    for c in range(cap,w-1,-1): dp[c]=max(dp[c],dp[c-w]+v)
    rows.append((f'w={w},v={v}',dp[:]))
show_table(['已处理物品','各容量最优值'],rows)

已处理物品,各容量最优值
无物品,"[0, 0, 0, 0, 0, 0, 0]"
"w=2,v=4","[0, 0, 4, 4, 4, 4, 4]"
"w=3,v=5","[0, 0, 4, 5, 5, 9, 9]"
"w=4,v=7","[0, 0, 4, 5, 7, 9, 11]"


## 7. 正确性、适用条件与复杂度

背包按“取第i件”与“不取”划分；逆序保证转移读取旧层。位集的第s位代表和s可达，左移x对应加入物品x；或操作合并两种可能。计数Counter每一步创建新层，零值对应两条不同决策边，所以自然翻倍。

**代价：** 背包O(nC)时间O(C)空间；位集在总和S位上做n次移位/或，位复杂度O(nS)上界、O(S)位空间；符号计数至多O(nS)状态处理，S=各数绝对值之和。这些是伪多项式代价，不是对输入比特长度的多项式。

### 展开理解

**背包为什么对**：对"第 i 件物品拿不拿"这个二选一，任何装法都必然落在"没拿它"或"拿了它"其中一侧，而且不可能同时落在两侧。"没拿它"的装法在容量 c 下的最好成绩，恰好等于"只用前 i−1 件、容量 c"的最优值；"拿了它"的装法去掉这件后是"只用前 i−1 件、容量 c−w"的某个装法，最好成绩就是那个格子的最优值加 v。两个格子取大，就覆盖了所有装法中的最优者。压缩成一维后，逆序遍历保证了读 `dp[c-w]` 时它还没被本层写过，等价于还在读上一层的格子，所以这个论证原封不动地成立。你可以做个对照实验：把 `range(capacity,w-1,-1)` 改成正序 `range(w,capacity+1)`，拿物品 w=2、v=4、容量 6 跑一遍，c=2 时 dp[2] 变 4，c=4 时 dp[4]=max(0, dp[2]+4)=8——同一件物品被算了两次，这就是练习 1 要你构造的反例。

**位集为什么对**：`reachable` 的第 s 位为 1，等价于"存在子集和恰为 s"，这个性质初始成立（只有空集，和为 0）。归纳地看：处理新物品 x 之后，"和 s 可达" ⟺ "不选 x 时 s 可达（原位是 1）"或"选 x 时 s−x 可达（左移前的第 s−x 位是 1）"。左移加按位或恰好把这两个条件逐位算出来，性质得以保持。

**计数为什么对**：`states[v]` 始终等于"前 k 个数的所有 2^k 种符号搭配中，表达式取值恰好为 v 的搭配数"。每个搭配的最后一步要么给第 k 个数添正号、要么添负号，把搭配按最后一步分成数量相加的两批，正是 `nxt[value±x]+=count` 做的事。

**复杂度**：背包是 n 件物品 × (C+1) 个容量格，每格常数操作，共 O(nC) 时间、O(C) 空间。比如 n=100 件物品、C=10⁴ 容量，约一百万次操作，一瞬间完成。位集在总和 S 个位上做 n 次移位/或，位操作次数上限 O(nS)，但 Python 大整数移位一次处理所有位，实际很快，占用 O(S) 个位。符号计数最多处理 O(nS) 个"值 × 方案"状态，其中 S 是各数绝对值之和。注意这三者都是伪多项式——步数跟"数值大小"走，不跟"输入文本长度"走，比如容量写成 10⁹，位数只有 30 位但格子有 10⁹ 个，就会爆。

## 8. 单元测试：每个用例在检查什么

```python
assert can_partition([1, 5, 11, 5])
```

- 正常值测试：总和 22、目标 11，子集 {11} 即可达成，验证位集可达性的基本正确性。

```python
assert knapsack_01([2], [3], 4) == 3
```

- 最小规模用例：单件物品、装得下（2≤4），答案就是它的价值 3，检验单物品转移和边界初始化。

```python
assert find_target_sum_ways([0, 0], 0) == 4
```

- 特殊值用例：两个 0 添正负号结果都不变，方案应当翻两番到 4。它专治一种常见错误——把 `+x` 和 `−x` 相同时（x=0）合并成一条转移，那样会错误地得到 2 或 1。

```python
for _ in range(140):
    n = rng.randrange(7)
    w = [rng.randrange(5) for _ in range(n)]
    v = [rng.randrange(-2, 9) for _ in range(n)]
    cap = rng.randrange(9)
    want = max((sum((v[i] for i in range(n) if mask >> i & 1)) for mask in range(1 << n) if sum((w[i] for i in range(n) if mask >> i & 1)) <= cap))
    assert knapsack_01(w, v, cap) == want
    assert can_partition(w) == any((2 * sum((w[i] for i in range(n) if mask >> i & 1)) == sum(w) for mask in range(1 << n)))
```

- 随机对拍：固定种子生成 140 组随机小实例（最多 6 件物品、重量 0..4、**价值允许为负** −2..8、容量 0..8），用 `mask` 枚举每个子集算暴力最优值，与 DP 结果逐一对照。注意价值有负数这个细节，它检验 `dp` 初始化为 0（允许什么都不装）时的"至多"语义是否与暴力一致；can_partition 同样用"存在子集两倍和等于总和"的枚举对照。

```python
for a in product(range(3), repeat=5):
    for target in range(-2, 3):
        assert find_target_sum_ways(a, target) == sum((sum((s * x for s, x in zip(signs, a))) == target for signs in product((-1, 1), repeat=5)))
```

- 穷举对拍：5 个 0..2 的数的所有组合 × 5 个目标值，暴力枚举 2⁵=32 种符号搭配计数，与 Counter DP 对照。0 与非 0 混合、正负目标都覆盖到了。

In [6]:
assert can_partition([1, 5, 11, 5])

assert knapsack_01([2], [3], 4) == 3

assert find_target_sum_ways([0, 0], 0) == 4

from itertools import product

from random import Random

rng = Random(101)

for _ in range(140):
    n = rng.randrange(7)
    w = [rng.randrange(5) for _ in range(n)]
    v = [rng.randrange(-2, 9) for _ in range(n)]
    cap = rng.randrange(9)
    want = max((sum((v[i] for i in range(n) if mask >> i & 1)) for mask in range(1 << n) if sum((w[i] for i in range(n) if mask >> i & 1)) <= cap))
    assert knapsack_01(w, v, cap) == want
    assert can_partition(w) == any((2 * sum((w[i] for i in range(n) if mask >> i & 1)) == sum(w) for mask in range(1 << n)))

for a in product(range(3), repeat=5):
    for target in range(-2, 3):
        assert find_target_sum_ways(a, target) == sum((sum((s * x for s, x in zip(signs, a))) == target for signs in product((-1, 1), repeat=5)))

print('N101: 所有本章断言通过')

N101: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 用正序更新构造重复拿物品反例。

**练习 2：** 推导目标和向子集计数的转换。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（正序反例）**：提示——把 `knapsack_01` 复制一份，容量循环改成从 w 到 capacity 的正序，然后选一个"单件物品就能重复占满背包"的输入，比如 `weights=[2]`、`values=[3]`、`capacity=6`，先手算逆序版本（答案 3，只能拿一次）和正序版本（dp 会一路传递到 dp[6]=9，等于拿了三件）各得多少，再跑代码验证你的预测。写清楚为什么正序时 `dp[c-w]` 里已经藏着当前物品。
- **练习 2（目标和转子集计数）**：提示——设添负号的数组成集合 N，表达式值 = 总和 − 2×(N 的和)，令它等于 target 就得到"N 的和必须等于 (S−target)/2"。于是问题变成"和恰好为某个定值的子集有几个"。注意推出两个前提：S−target 必须是偶数、且非负，否则方案数为 0。用一个小例子（如 `[1,1,1,1,1]`、target=3，答案 5）先手算子集计数，再和 `find_target_sum_ways` 的结果互相验证。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from collections import Counter

def knapsack_01(weights, values, capacity):
    if len(weights) != len(values) or capacity < 0 or any((w < 0 for w in weights)):
        raise ValueError('invalid knapsack')
    dp = [0] * (capacity + 1)
    for w, v in zip(weights, values):
        for c in range(capacity, w - 1, -1):
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[-1]

def can_partition(nums):
    total = sum(nums)
    if any((x < 0 for x in nums)):
        raise ValueError('nonnegative values required')
    if total % 2:
        return False
    reachable = 1
    for x in nums:
        reachable |= reachable << x
    return bool(reachable >> total // 2 & 1)

def find_target_sum_ways(nums, target):
    states = Counter({0: 1})
    for x in nums:
        nxt = Counter()
        for value, count in states.items():
            nxt[value + x] += count
            nxt[value - x] += count
        states = nxt
    return states[target]

# 示例与回归测试
assert can_partition([1, 5, 11, 5])

assert knapsack_01([2], [3], 4) == 3

assert find_target_sum_ways([0, 0], 0) == 4

from itertools import product

from random import Random

rng = Random(101)

for _ in range(140):
    n = rng.randrange(7)
    w = [rng.randrange(5) for _ in range(n)]
    v = [rng.randrange(-2, 9) for _ in range(n)]
    cap = rng.randrange(9)
    want = max((sum((v[i] for i in range(n) if mask >> i & 1)) for mask in range(1 << n) if sum((w[i] for i in range(n) if mask >> i & 1)) <= cap))
    assert knapsack_01(w, v, cap) == want
    assert can_partition(w) == any((2 * sum((w[i] for i in range(n) if mask >> i & 1)) == sum(w) for mask in range(1 << n)))

for a in product(range(3), repeat=5):
    for target in range(-2, 3):
        assert find_target_sum_ways(a, target) == sum((sum((s * x for s, x in zip(signs, a))) == target for signs in product((-1, 1), repeat=5)))

print('N101: 所有本章断言通过')

N101: 所有本章断言通过


## 11. 代表题与迁移

- **416. Partition Equal Subset Sum（分割等和子集）**：把"分成相等的两堆"转成"子集和恰好为总和一半"的可达性问题，对应 `can_partition` 的位集解法。
- **494. Target Sum（目标和）**：符号计数 DP 的原型题，练"方案数"与"最优值"这两种 DP 输出的区别，对应 `find_target_sum_ways`。
- **1049. Last Stone Weight II（最后一块石头的重量 II）**：本质是"把石头分成两堆使差最小"，可转成"子集和尽量接近总和一半"的 0/1 背包可达性问题，是 416 的近亲迁移。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。